# EADM revision runs (3 seeds, shared teacher, leave-one-out ablation on seed 42)

Runtime → Change runtime type → **GPU** first. Everything is written to Google Drive, so if Colab disconnects just run the cells again; finished checkpoints are skipped.

**Before running: upload the latest `ubmec_fixed_size_breakpoint_experiment.py` and `run_ubmec_multiseed_study.py` to `MyDrive/eadm/` (replace the old copies).**

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
%mkdir -p /content/drive/MyDrive/eadm
%cd /content/drive/MyDrive/eadm
!ls -la

## Step 2: files
Before continuing, make sure these 4 files are inside `MyDrive/eadm/`:
`UBMEC.csv`, `ubmec_fixed_size_breakpoint_experiment.py`, `run_ubmec_multiseed_study.py`, `eadm_revision_analysis.py`

In [ ]:
import os
for f in ['UBMEC.csv','ubmec_fixed_size_breakpoint_experiment.py','run_ubmec_multiseed_study.py','eadm_revision_analysis.py']:
    print(('OK   ' if os.path.exists(f) else 'MISSING ')+f)

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf scipy scikit-learn pandas matplotlib tqdm

## Step 3: validation, seed 42 with leave-one-out ablation (~2 h on a T4)

In [ ]:
!python run_ubmec_multiseed_study.py validation --seeds 42 --ablation --ablation-seeds 42 --ablation-mode leave_one_out

## Step 4: validation, seeds 123 and 2026 (teacher reused from seed 42, no ablation; ~45 min each). Re-run after any disconnect.

In [ ]:
!python run_ubmec_multiseed_study.py validation --seeds 123 2026 --ablation --ablation-seeds 42 --ablation-mode leave_one_out --shared-teacher-seed 42

## Step 5: final test evaluation + aggregate for the 3 seeds (~20-30 min)

In [ ]:
!python run_ubmec_multiseed_study.py final --seeds 42 123 2026 --ablation --ablation-seeds 42 --ablation-mode leave_one_out --shared-teacher-seed 42
!python run_ubmec_multiseed_study.py aggregate --seeds 42 123 2026 --ablation --ablation-seeds 42 --ablation-mode leave_one_out --shared-teacher-seed 42

## Step 6: revision artefacts (latency, confusion matrices, per-class, case-study candidates, significance)

In [ ]:
import glob, json
run42 = None
for meta in glob.glob('ubmec_professor_multiseed_outputs/live/*/run_metadata.json'):
    m = json.load(open(meta))
    print(meta, 'seed', m.get('training_seed'))
    if m.get('training_seed') == 42:
        run42 = meta.rsplit('/', 1)[0]
print('seed-42 run dir:', run42)

In [ ]:
!python eadm_revision_analysis.py --run-dir "$run42" --multiseed-aggregate ubmec_professor_multiseed_outputs/aggregate --out revision_artefacts

## Step 7: zip everything Claude needs, then download `eadm_outputs.zip` from the Files panel (or from Drive)

In [ ]:
!rm -f eadm_outputs.zip
!zip -r eadm_outputs.zip revision_artefacts ubmec_professor_multiseed_outputs/aggregate ubmec_professor_multiseed_outputs/multiseed_protocol_plan.json ubmec_professor_multiseed_outputs/live/*/fixed_size_breakpoint_results.csv ubmec_professor_multiseed_outputs/live/*/ablation_test_results.csv ubmec_professor_multiseed_outputs/live/*/run_metadata.json ubmec_professor_multiseed_outputs/live/*/precision_parity_validation.csv ubmec_professor_multiseed_outputs/live/*/model_artifact_audit.csv ubmec_professor_multiseed_outputs/live/*/deployment_audit_internal.csv ubmec_professor_multiseed_outputs/live/*/ablation_validation_results.csv
!ls -la eadm_outputs.zip
from google.colab import files
files.download('eadm_outputs.zip')